# Top n de documentos recuperados por modelo (consulta de prueba)

Dado un **número de consulta** de la partición de prueba de MessIRve, este cuaderno devuelve los **n documentos mejor posicionados** (10 por omisión) de cada **modelo de recuperación inicial** de los experimentos, en el orden en que ese modelo los rankea.

Los rankings se leen de los *runs* que ya calculó el pipeline de recuperación (`~/.cache/messirve_embeddings/<modelo>/<configuración>/retrieval_run.lz4`): el cuaderno **no vuelve a recuperar** nada, no codifica embeddings ni usa GPU. Se ejecuta con el entorno `proyecto` (conda), que es el que tiene `ranx`, `datasets` y `pandas`.

**Modelos incluidos** (seis, ver §2): BM25, SPLADE-v3, multilingual-e5-large-instruct, BGE-M3, Qwen3-Embedding-0.6B y jina-embeddings-v5-text-small-retrieval. `microsoft/harrier-oss-v1-0.6b` **queda excluido** a propósito.

**Uso.** Fija `NUMERO_CONSULTA` en la §1 (es el `id` numérico de la consulta en el split de prueba; el ejemplo es `8101866`) y ejecuta el cuaderno completo. Para inspeccionar **otras** consultas está la **«Versión interactiva»** del final, con `analizar_consulta(numero, n)`, que recupera, carga los textos y muestra las tablas de la consulta que se le pase sin volver a ejecutar el resto; para *encontrar* un número están `listar_consultas()` y `buscar_consultas()` en la §3.

## 1. Configuración

- `NUMERO_CONSULTA`: número (`id`) de la consulta de prueba que se quiere inspeccionar.
- `N`: cuántos documentos devuelve cada modelo.
- `INCLUIR_TEXTO`: si es `True`, la §5 lee el corpus de párrafos de la Wikipedia en español para añadir el título y el texto de cada documento; si es `False`, las tablas salen solo con `docid` y puntaje.
- `EXTRACTO_CARACTERES`: cuántos caracteres del documento se muestran en la columna `extracto` de las tablas. Los párrafos que devuelve la recuperación rondan los 500 caracteres y casi nunca pasan de 1600, así que con 2000 se leen completos; `None` quita el tope.

Estos valores son los de la consulta de ejemplo; la «Versión interactiva» del final analiza otras consultas con `analizar_consulta(numero, n)`.

In [1]:
import gc
import sys
import time
from dataclasses import dataclass
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

# ---------------------------------------------------------------------------
# Configuración
# ---------------------------------------------------------------------------
NUMERO_CONSULTA = 8101866   # número (id) de la consulta de prueba a inspeccionar; ver §3
N = 10                      # documentos que devuelve cada modelo
INCLUIR_TEXTO = True        # True: título y extracto de cada documento (lee el corpus, §5)
EXTRACTO_CARACTERES = 2000  # caracteres de extracto por documento en las tablas (None: completo)

# Las tablas de la §6 llevan el texto de cada documento en la columna `extracto`, y pandas recorta
# las celdas a 50 caracteres por omisión (tanto en el HTML como en el texto): se desactiva el
# recorte para que el extracto se lea entero.
pd.set_option("display.max_colwidth", None)


# ---------------------------------------------------------------------------
# Entorno
# ---------------------------------------------------------------------------
def _raiz_repositorio() -> Path:
    """Carpeta de ir-spanish/ (la que contiene utils/), para importar el código del pipeline.

    El cuaderno vive en analysis/, pero JupyterLab lo abre con ese directorio como cwd y
    nbconvert lo abre donde se lance, así que la raíz se busca hacia arriba en vez de suponerla.
    """
    for base in (Path.cwd(), *Path.cwd().parents):
        if (base / "utils" / "cache.py").exists():
            return base
    raise RuntimeError(
        "No se encontró la raíz de ir-spanish (la carpeta que contiene utils/). "
        "Abre el cuaderno desde el repositorio o ejecútalo desde su raíz."
    )


RAIZ = _raiz_repositorio()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from ranx.io import load_lz4          # lee un run tal como lo escribió ranx.Run.save
from utils import cache, data         # rutas de la caché y constantes del conjunto de datos

# `utils` deja el registro en nivel INFO y huggingface_hub registra cada petición HTTP, lo que
# llena de ruido la salida al leer el corpus (§5). Se silencian solo esos registros de terceros.
import logging

logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)

CACHE_DIR = Path.home() / ".cache" / "messirve_embeddings"

print(f"Repositorio:   {RAIZ}")
print(f"Caché de runs: {CACHE_DIR}")
print(f"Consulta:      {NUMERO_CONSULTA}   |   n = {N}   |   texto de los documentos: {INCLUIR_TEXTO}")

/home/jmendoza/miniconda3/envs/proyecto/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Repositorio:   /home/jmendoza/ir-spanish
Caché de runs: /home/jmendoza/.cache/messirve_embeddings
Consulta:      8101866   |   n = 10   |   texto de los documentos: True


## 2. Modelos de recuperación inicial

Cada modelo tiene su *run* cacheado. La ruta se deriva con `utils.cache` —la misma que usa el pipeline— a partir de la configuración con la que se generó el run (`max_query_length` y `max_doc_length` de la tabla de resultados). BM25 no tiene longitudes de secuencia, así que `baselines/bm25.py` guarda su run en el directorio que solo depende del conjunto de datos; el mismo archivo está también en la ruta con longitudes (idénticos byte a byte), y se acepta cualquiera de las dos.

`microsoft/harrier-oss-v1-0.6b` **no se incluye** (petición del usuario, 2026-09-25). Queda en el registro como línea comentada, para que la exclusión sea explícita y no un olvido.

In [2]:
@dataclass(frozen=True)
class ModeloInicial:
    """Un modelo de recuperación inicial y la configuración con la que se cacheó su run."""
    nombre: str         # nombre en HuggingFace o identificador del pipeline
    alias: str          # nombre corto para las tablas
    max_q: int | None   # max_query_length del run (None: el modelo no trunca por longitudes)
    max_d: int | None   # max_doc_length del run


MODELOS = [
    # Léxico
    ModeloInicial("bm25_pyserini", "bm25", None, None),
    # Disperso (learned sparse)
    ModeloInicial("naver/splade-v3", "splade-v3", 512, 512),
    # Densos (dual-encoders)
    ModeloInicial("intfloat/multilingual-e5-large-instruct", "e5-large", 512, 512),
    ModeloInicial("BAAI/bge-m3", "bge-m3", 8192, 8192),
    ModeloInicial("Qwen/Qwen3-Embedding-0.6B", "qwen3-0.6b", 32768, 32768),
    ModeloInicial("jinaai/jina-embeddings-v5-text-small-retrieval", "jina-v5-small", 32768, 32768),
    # Excluido a propósito del análisis (2026-09-25):
    # ModeloInicial("microsoft/harrier-oss-v1-0.6b", "harrier", 32768, 32768),
]


def rutas_run(modelo: ModeloInicial) -> list[Path]:
    """Rutas donde puede estar el run de un modelo, en orden de preferencia.

    BM25 no tiene longitudes de secuencia: `baselines/bm25.py` guarda su run en el directorio
    que solo depende del conjunto de datos, y el mismo archivo se copió a la ruta con longitudes,
    que es la que usan los scripts de fusión. Se acepta cualquiera de las dos.
    """
    if modelo.max_q is None:
        solo_conjunto = CACHE_DIR / cache.model_slug(modelo.nombre) / (
            f"{data.COUNTRY}_v{data.DATASET_VERSION}_{cache._filter_suffix(data.MAX_WORD_COUNT)}"
        )
        con_longitudes = cache.cache_base(
            CACHE_DIR, modelo.nombre, data.COUNTRY, data.DATASET_VERSION,
            512, 512, data.MAX_WORD_COUNT,
        )
        return [cache.run_cache_path(solo_conjunto), cache.run_cache_path(con_longitudes)]

    base = cache.cache_base(
        CACHE_DIR, modelo.nombre, data.COUNTRY, data.DATASET_VERSION,
        modelo.max_q, modelo.max_d, data.MAX_WORD_COUNT,
    )
    return [cache.run_cache_path(base)]


def ruta_run(modelo: ModeloInicial) -> Path:
    """Primera ruta existente del run; falla con un mensaje claro si no hay ninguna."""
    candidatas = rutas_run(modelo)
    for ruta in candidatas:
        if ruta.exists():
            return ruta
    raise FileNotFoundError(
        f"No hay run cacheado para {modelo.nombre}. Rutas probadas:\n  "
        + "\n  ".join(str(ruta) for ruta in candidatas)
    )


# Comprobación: de qué archivo se va a leer cada modelo
filas = []
for modelo in MODELOS:
    run = ruta_run(modelo)
    filas.append({
        "alias": modelo.alias,
        "modelo": modelo.nombre,
        "max_query_length, max_doc_length": "—" if modelo.max_q is None else f"{modelo.max_q}, {modelo.max_d}",
        "run": str(run.relative_to(CACHE_DIR)),
        "MB": round(run.stat().st_size / 1024**2, 1),
    })

display(pd.DataFrame(filas))

,alias,modelo,"max_query_length, max_doc_length",run,MB
0,bm25,bm25_pyserini,—,bm25_pyserini/full_v1.2_nofilter/retrieval_run.lz4,144.1
1,splade-v3,naver/splade-v3,"512, 512",naver__splade-v3/full_v1.2_q512_d512_nofilter/retrieval_run.lz4,159.1
2,e5-large,intfloat/multilingual-e5-large-instruct,"512, 512",intfloat__multilingual-e5-large-instruct/full_v1.2_q512_d512_nofilter/retrieval_run.lz4,148.1
3,bge-m3,BAAI/bge-m3,"8192, 8192",BAAI__bge-m3/full_v1.2_q8192_d8192_nofilter/retrieval_run.lz4,154.2
4,qwen3-0.6b,Qwen/Qwen3-Embedding-0.6B,"32768, 32768",Qwen__Qwen3-Embedding-0.6B/full_v1.2_q32768_d32768_nofilter/retrieval_run.lz4,154.5
5,jina-v5-small,jinaai/jina-embeddings-v5-text-small-retrieval,"32768, 32768",jinaai__jina-embeddings-v5-text-small-retrieval/full_v1.2_q32768_d32768_nofilter/retrieval_run.lz4,154.1


## 3. La consulta de prueba y su verdad de referencia

El **número de consulta** es el campo `id` del split de prueba (un entero, p. ej. `8101866`); las claves de los runs y de la verdad de referencia son ese número en forma de cadena. La partición de prueba tiene 170,055 consultas únicas.

La verdad de referencia se lee de `pruned_qrels.json`, el mismo archivo con el que el pipeline evalúa: para cada consulta, los `docid` relevantes. Un documento es un párrafo de un artículo de Wikipedia, identificado por un `docid` de la forma `"articulo#parrafo"` (p. ej. `"328242#0"`).

Para elegir un número: `listar_consultas(inicio, cuantas)` recorre la partición y `buscar_consultas("fragmento")` filtra por el texto de la consulta.

In [3]:
# Verdad de referencia y mapa consulta → texto, del caché compartido que usa el pipeline
dataset_cache_dir = cache.dataset_cache_base(
    CACHE_DIR, data.COUNTRY, data.DATASET_VERSION, data.MAX_WORD_COUNT
)
qrels, consulta_a_texto = data.get_pruned_qrels_and_queries(
    data.COUNTRY, data.DATASET_VERSION, kept_doc_ids=None,
    dataset_cache_dir=dataset_cache_dir, num_workers=1,
)
relevantes_por_consulta = qrels.to_dict()

print(f"Consultas de prueba: {len(consulta_a_texto):,}")
print(f"Verdad de referencia: {dataset_cache_dir / 'pruned_qrels.json'}")


def resolver_consulta(numero) -> str:
    """Devuelve el id de consulta (cadena) que corresponde al número dado.

    Se acepta como entero o como cadena; las claves de los runs y de los qrels son cadenas.
    """
    numero = str(numero)
    if numero not in consulta_a_texto:
        raise KeyError(
            f"La consulta {numero!r} no está en la partición de prueba. "
            "Usa listar_consultas() o buscar_consultas('...') para encontrar un número válido."
        )
    return numero


def listar_consultas(inicio: int = 0, cuantas: int = 10) -> None:
    """Imprime una rebanada de la partición, con el número y el texto de cada consulta."""
    for numero in list(consulta_a_texto)[inicio:inicio + cuantas]:
        print(f"{numero}  {consulta_a_texto[numero]}")


def buscar_consultas(fragmento: str, limite: int = 20) -> None:
    """Imprime las consultas cuyo texto contiene `fragmento` (sin distinguir mayúsculas)."""
    fragmento = fragmento.lower()
    encontradas = 0
    for numero, texto in consulta_a_texto.items():
        if fragmento in texto.lower():
            print(f"{numero}  {texto}")
            encontradas += 1
            if encontradas == limite:
                break
    print(f"\n{encontradas} consulta(s) mostradas (límite {limite}).")


# Ejemplos de uso para encontrar un número de consulta
listar_consultas(0, 5)
buscar_consultas("moneda circula en aruba")

15:35:39 | INFO | ⏱  START: Loading cached pruned qrels


15:35:39 | INFO | ⏱  DONE:  Loading cached pruned qrels (0.3s)


Consultas de prueba: 170,055
Verdad de referencia: /home/jmendoza/.cache/messirve_embeddings/shared_datasets/full_v1.2_nofilter/pruned_qrels.json
7397859   en grecia quién aplico la democracia radical
7397860   que conoces de la familia arduino
7397866  1 arroba cuantas kilogramos tiene
7397867  1 arroba cuantos kg tiene
7397868  1 arroba cuántas libras tiene
8101866  que moneda circula en aruba

1 consulta(s) mostradas (límite 20).


In [4]:
# Consulta configurada en la §1
CONSULTA = resolver_consulta(NUMERO_CONSULTA)
print(f"Consulta {CONSULTA}: {consulta_a_texto[CONSULTA]}")
for docid in relevantes_por_consulta[CONSULTA]:
    print(f"  relevante: {docid}")

Consulta 8101866: que moneda circula en aruba
  relevante: 328242#0


## 4. Top n por modelo

Se lee el run de cada modelo (ranx lo guarda como `{consulta: {docid: score}}`), se ordenan los documentos de esa consulta por score descendente —con orden estable, para respetar el orden del run cuando hay empates— y se toman los `n` primeros. Cada run se libera antes de cargar el siguiente, de modo que no están los seis rankings en memoria a la vez.

Leer un run cuesta unos segundos (≈150 MB comprimidos, ≈2.5 GB en memoria) y se repite en cada llamada: los seis modelos tardan ≈30 s, así que conviene tener claro qué consultas se quieren inspeccionar antes de ejecutar la celda varias veces.

In [5]:
def top_n_por_modelo(numero, n: int = N) -> dict[str, pd.DataFrame]:
    """Devuelve, por modelo, el top n de documentos recuperados para una consulta de prueba.

    Returns:
        {alias: DataFrame} con columnas: posicion, docid, puntaje, relevante.
    """
    numero = resolver_consulta(numero)
    relevantes = relevantes_por_consulta[numero]
    resultados: dict[str, pd.DataFrame] = {}

    for modelo in MODELOS:
        ruta = ruta_run(modelo)
        t0 = time.time()
        run = load_lz4(str(ruta))       # mismo dict que escribió ranx.Run.save
        if numero not in run:
            raise KeyError(f"El run de {modelo.alias} no contiene la consulta {numero}: {ruta}")

        top = sorted(run[numero].items(), key=lambda par: par[1], reverse=True)[:n]
        del run
        gc.collect()

        resultados[modelo.alias] = pd.DataFrame([
            {
                "posicion": posicion,
                "docid": docid,
                "puntaje": puntaje,
                "relevante": "sí" if docid in relevantes else "",
            }
            for posicion, (docid, puntaje) in enumerate(top, start=1)
        ])

        aviso = "" if len(top) == n else f"  (el run solo devolvió {len(top)})"
        print(f"{modelo.alias:>13}: {len(top):2d} documentos en {time.time() - t0:.1f} s{aviso}")

    return resultados

In [6]:
# Top n de la consulta configurada en la §1 (la que resolvió la §3). Para otra consulta, usa la
# «Versión interactiva» del final:  analizar_consulta(7521003, N)
resultados = top_n_por_modelo(CONSULTA, N)

         bm25: 10 documentos en 4.9 s


    splade-v3: 10 documentos en 4.8 s


     e5-large: 10 documentos en 4.8 s


       bge-m3: 10 documentos en 4.7 s


   qwen3-0.6b: 10 documentos en 4.7 s


jina-v5-small: 10 documentos en 4.8 s


## 5. Textos de los documentos

`cargar_textos(resultados, numero)` busca en el corpus de párrafos de la Wikipedia en español (`eswiki_20240401`) los `docid` que aparecen en las tablas —los recuperados y los relevantes— y devuelve su título y su texto. El corpus está en la caché de HuggingFace y se lee mapeado en disco: con una máscara de pyarrow se materializan solo los párrafos pedidos (≈40), no los 14 millones del corpus.

Los textos son **de una consulta**: al cambiar de consulta hay que volver a llamar a la función (lo hace `analizar_consulta` en la «Versión interactiva»). Si `INCLUIR_TEXTO = False`, no se carga nada y las tablas salen sin título ni extracto.

In [7]:
import datasets
import pyarrow as pa
import pyarrow.compute as pc


def cargar_textos(resultados, numero) -> dict[str, tuple[str, str]]:
    """Título y texto de los párrafos de una consulta: los recuperados y los relevantes.

    Returns:
        {docid: (título, texto)}; diccionario vacío si INCLUIR_TEXTO es False.
    """
    if not INCLUIR_TEXTO:
        print("INCLUIR_TEXTO = False: las tablas salen sin título ni extracto.")
        return {}

    docids = sorted(
        set().union(*[set(tabla["docid"]) for tabla in resultados.values()])
        | set(relevantes_por_consulta[numero])
    )

    t0 = time.time()
    corpus = datasets.load_dataset(data.CORPUS_NAME, split="corpus")
    tabla_corpus = corpus.data.table
    columna_docid = pc.cast(tabla_corpus["docid"], pa.string())
    seleccion = tabla_corpus.filter(
        pc.is_in(columna_docid, value_set=pa.array(docids, type=columna_docid.type))
    )

    textos: dict[str, tuple[str, str]] = {}
    for docid, titulo, texto in zip(
        seleccion["docid"].to_pylist(),
        seleccion["title"].to_pylist(),
        seleccion["text"].to_pylist(),
    ):
        textos[str(docid)] = (titulo or "", texto or "")

    print(f"Párrafos pedidos: {len(docids)} | encontrados en el corpus: {len(textos)} "
          f"({time.time() - t0:.1f} s)")
    return textos


# Textos de la consulta configurada en la §1. Son de una consulta: al cambiar de consulta hay
# que volver a llamar a la función (lo hace `analizar_consulta` en la «Versión interactiva»).
textos = cargar_textos(resultados, CONSULTA)

Párrafos pedidos: 38 | encontrados en el corpus: 38 (1.7 s)


## 6. Resultados

El top n de cada modelo para la consulta configurada. `posicion` es el lugar que el modelo le da al documento (1 = primero), `puntaje` es el score del run y `relevante` marca los documentos que la verdad de referencia considera relevantes para esta consulta.

La columna `extracto` trae el texto del párrafo hasta `EXTRACTO_CARACTERES` (ver §1) y la tabla no lo recorta, así que el documento se lee sin salir del cuaderno.

`mostrar_resultados(resultados, numero, textos)` y `mostrar_relevantes(numero, textos)` reciben la consulta, así que sirven para cualquier número (es lo que usa la «Versión interactiva»). Después de las tablas se imprime el **texto completo de los documentos relevantes**, para poder juzgar lo que cada modelo devolvió.

In [8]:
def _extracto(texto: str, limite: int | None = EXTRACTO_CARACTERES) -> str:
    """Texto del documento para la tabla, recortado a `limite` caracteres (None: sin tope)."""
    texto = " ".join(texto.split())
    if limite is None or len(texto) <= limite:
        return texto
    return texto[:limite].rsplit(" ", 1)[0] + " […]"


def mostrar_resultados(resultados, numero, textos) -> None:
    """Muestra el top n de cada modelo, con título y extracto si hay textos cargados.

    Recibe la consulta (`numero`) y sus `textos` en vez de leerlos de variables globales: así
    sirve para cualquier consulta, no solo para la configurada en la §1.
    """
    for alias, tabla in resultados.items():
        vista = tabla.copy()
        if textos:
            vista["titulo"] = [textos.get(docid, ("", ""))[0] for docid in vista["docid"]]
            vista["extracto"] = [_extracto(textos.get(docid, ("", ""))[1]) for docid in vista["docid"]]
        display(Markdown(f"**{alias}** — top {len(vista)} de «{consulta_a_texto[numero]}»"))
        display(vista)


def mostrar_relevantes(numero, textos) -> None:
    """Imprime el texto completo de los documentos relevantes de una consulta."""
    display(Markdown("**Documentos relevantes (verdad de referencia)**"))
    for docid in relevantes_por_consulta[numero]:
        titulo, texto = textos.get(docid, ("(sin texto)", ""))
        print(f"{docid} — {titulo}\n{texto}\n")


# Resultados de la consulta configurada en la §1
mostrar_resultados(resultados, CONSULTA, textos)
mostrar_relevantes(CONSULTA, textos)

**bm25** — top 10 de «que moneda circula en aruba»

,posicion,docid,puntaje,relevante,titulo,extracto
0,1,10403#24,10.3992,,Aruba,"Aruba es parte del Reino de los Países Bajos, pero mantiene amplio control sobre sus asuntos excepto cuestiones relacionadas con la defensa nacional, ciudadanía, relaciones exteriores y extradición. Aruba posee sus propias leyes, constitución, gobierno y moneda oficial."
1,2,15756#0,10.3215,,Gobierno y política de Aruba,"Aruba es parte del Reino de los Países Bajos, pero mantiene total control sobre sus asuntos excepto cuestiones relacionadas con la defensa nacional, ciudadanía, relaciones exteriores y extradición. Aruba posee sus propias leyes, constitución, gobierno y moneda oficial."
2,3,943022#4,9.7543,,Florín antillano neerlandés,"En 1986, Aruba ganó un estatus de autonomía y se separó de las Antillas Neerlandesas. En poco tiempo, Aruba comenzó a emitir su propia moneda, el florín arubeño, que sustituyó al florín de las Antillas Neerlandesas a un cambio de 1:1."
3,4,1992977#7,9.6735,,Unión monetaria de América del Norte,"Existirían antecedentes de esa Unión. Se han practicado niveles menores de cooperación monetaria continental anteriormente. Algunas naciones, tales como Argentina, Brasil y Canadá, en ocasiones han pareado su moneda al dólar estadounidense, con resultados controvertidos (en Argentina con resultados muy malos para la mayoría de la población). Algunas otras, como Aruba, las Bahamas y Barbados todavía lo hacen. Así mismo, el dólar estadounidense es aceptado oficialmente junto con monedas locales en El Salvador (desde 2001), Nicaragua, Perú, Honduras y Panamá, aunque, en la práctica, dos de estos países (El Salvador y Panamá) están completamente dolarizados. En el 2000, Ecuador adoptó oficialmente el dólar estadounidense como su moneda única. En un caso particular Guatemala creó una ley que llama Ley de Libre Negociación de Divisas, vigente desde el 1 de mayo de 2001, que consiste en poner al dólar estadounidense (junto a otras monedas del mundo), como monedas de uso legal, mas no de curso legal, pues el quetzal aún circula como la única moneda de curso legal. En el caso de esta ley, le permite a cualquier persona, negociar bienes o abrir cuentas bancarias en otro tipo de moneda, siendo el dólar estadounidense la más usada. En la Argentina, durante el gobierno de Carlos Menem, se sancionó una ley, inspirada por el ministro de economía Domingo Cavallo, que estipulaba la convertibilidad automática de la moneda argentina (en ese momento el Austral, más tarde el peso convertible) con el dólar estadounidense. Esta ley dejó de estar en vigencia en 2002, después de una terrible crisis económica y política."
4,5,5585611#0,9.6443,,Museo numismático de Aruba,"El Museo numismático de Aruba es un museo establecido en la ciudad de Oranjestad, la capital del país autónomo de los Países Bajos e isla caribeña de Aruba. Se trata de un espacio especializado que cuenta con una gran colección de monedas y billetes de todo el mundo, y que representa la pasión de un hombre local (Mario Odor) cuyo afición se transformó en una exhibición para el público. Fue creado oficialmente el 13 de noviembre de 1981, posee 35 piezas que vienen de 400 lugares."
5,6,8895256#8,9.0573,,Paradoja de las ruedas de Aristóteles,"El autor de ""Falacias y paradojas matemáticas"" usa una moneda de diez centavos pegada a medio dólar con sus centros alineados, ambos fijos a un eje, como modelo para la paradoja. La moneda de diez centavos sirve como el círculo más pequeño y el medio dólar como el más grande. El escribe: Esta es la solución, entonces, o la clave para ello. Aunque tiene cuidado de no dejar que el medio dólar se deslice sobre la mesa, el ""punto"" que rastrea el segmento de línea al pie de la moneda de diez centavos gira y se desliza todo el tiempo. Se está deslizando con respecto a la mesa. Como la moneda de diez centavos no toca la parte superior de la mesa, no notará el deslizamiento. Si puede mover el medio dólar a lo largo de la mesa y al mismo tiempo hacer rodar

**splade-v3** — top 10 de «que moneda circula en aruba»

,posicion,docid,puntaje,relevante,titulo,extracto
0,1,1992977#7,18.231495,,Unión monetaria de América del Norte,"Existirían antecedentes de esa Unión. Se han practicado niveles menores de cooperación monetaria continental anteriormente. Algunas naciones, tales como Argentina, Brasil y Canadá, en ocasiones han pareado su moneda al dólar estadounidense, con resultados controvertidos (en Argentina con resultados muy malos para la mayoría de la población). Algunas otras, como Aruba, las Bahamas y Barbados todavía lo hacen. Así mismo, el dólar estadounidense es aceptado oficialmente junto con monedas locales en El Salvador (desde 2001), Nicaragua, Perú, Honduras y Panamá, aunque, en la práctica, dos de estos países (El Salvador y Panamá) están completamente dolarizados. En el 2000, Ecuador adoptó oficialmente el dólar estadounidense como su moneda única. En un caso particular Guatemala creó una ley que llama Ley de Libre Negociación de Divisas, vigente desde el 1 de mayo de 2001, que consiste en poner al dólar estadounidense (junto a otras monedas del mundo), como monedas de uso legal, mas no de curso legal, pues el quetzal aún circula como la única moneda de curso legal. En el caso de esta ley, le permite a cualquier persona, negociar bienes o abrir cuentas bancarias en otro tipo de moneda, siendo el dólar estadounidense la más usada. En la Argentina, durante el gobierno de Carlos Menem, se sancionó una ley, inspirada por el ministro de economía Domingo Cavallo, que estipulaba la convertibilidad automática de la moneda argentina (en ese momento el Austral, más tarde el peso convertible) con el dólar estadounidense. Esta ley dejó de estar en vigencia en 2002, después de una terrible crisis económica y política."
1,2,2476004#45,17.676958,,Moneda (divisa),"Cuando la letra circula, el que circula es simplemente un papel que representa una prometida de pago en metálico a una cierta fecha, pero este metálico todavía no existe; por lo tanto, la letra de cambio no sustituye la moneda metálica, sino que se añade; es un nuevo instrumento monetario que, además, no tiene ningún valor en él mismo, sino únicamente el de la confianza que puede inspirar en que el pago será realmente efectuado un golpe el plazo esté completo."
2,3,1096476#3,17.478882,,Dinar argelino,"Las monedas que circulan con mayor asiduidad son las de 5 dinares o de valor superior. Debido a la inflación sufrida por el país en la transición a una economía capitalista a principios de los 90, los céntimos o francciones de dinar fueron desechados de la circulación, mientras que las monedas de 1 y 2 dinares son raramente usadas. Sin embargo, los precios siguen mostrándose en céntimos, y se siguen usando en las conversaciones diarias, hasta el punto que un precio de 100 dinares se lee ""diez mil"" (عشر الاف). En agosto de 2012 fue puesta en circulación una nueva moneda bimetálica de 200 dinares en conmemoración del 50 aniversario de la independencia."
3,4,10403#135,16.608856,,Aruba,"El 19 de febrero de 2013, Arubus puso en marcha su primera línea de tranvía. El tranvía de Aruba, denominado oficialmente ""Arutram"", (""Tram van Oranjestad, Oranjestad Streetcar)"" circula cada ocho minutos y medio entre la terminal de cruceros de las afueras de Oranjestad y el centro de la ciudad. Los tres vehículos sobre rieles abiertos, que utilizan hidrógeno como fuente de energía, funcionan con pilas de combustible. La energía necesaria se genera a partir de la energía solar y eólica. El tranvía no necesita cables aéreos. Los vehículos, de aspecto histórico, fueron construidos por TIG/m-LLC, con sede en Chatsworth (EE. UU.). Los rieles acanalados colocados en el asfalto fueron fabricados por la empresa TSTG, con sede en Duisburgo (Alemania)."
4,5,328242#0,16.016756,sí,Florín arubeño,"El florín es la moneda oficial de Aruba. Se divide en 100 céntimos. Fue introducido en 1986, reemplazando a paridad el Florín antillano neerlandés."
5,6,2472607#19,15.500867,,Monedas del reino visigodo,"Se han contabi

**e5-large** — top 10 de «que moneda circula en aruba»

...

,posicion,docid,puntaje,relevante,titulo,extracto
0,1,328242#0,0.778951,sí,Florín arubeño,"El florín es la moneda oficial de Aruba. Se divide en 100 céntimos. Fue introducido en 1986, reemplazando a paridad el Florín antillano neerlandés."
1,2,7733876#2,0.733029,,Banco Central de Aruba,"El banco es una entidad legal en sí misma (""sui generis"") con una posición autónoma dentro del sector público de Aruba. Con la creación del banco, el florín arubeño fue puesto en circulación al mismo tipo de cambio que el florín antillano neerlandés, ligado al dólar estadounidense a Afl. 1.79 (= 1 NAf.) = US$1.00. Este tipo de cambio se ha mantenido sin cambios desde entonces."
2,3,10403#54,0.724236,,Aruba,"Alrededor del 70 % del PIB de Aruba proviene del turismo o de actividades relacionadas, y un 75 % de los visitantes procede de Estados Unidos. Antes de obtener su estatus autonómico, la mayor actividad era la refinación de petróleo; actualmente dicha actividad tiene una pequeña influencia en la economía. La agricultura y la manufactura tienen también un pequeño impacto económico. El florín arubiano generalmente tiene un cambio fijo con el dólar estadounidense de 1,75:1, pero en la mayoría de los comercios se aplica 1,80:1 como tasa de cambio. Sus principales socios comerciales son Venezuela, Estados Unidos, Países Bajos y Reino Unido."
3,4,3655473#0,0.696541,,Economía de Aruba,"La Economía de Aruba es de libre mercado. El turismo constituye en la actualidad la base del mayor porcentaje de los ingresos del país. Por el rápido crecimiento del turismo en los últimos 80 años, las industrias relacionadas, como la construcción también han florecido. Otras industrias principales incluyen el refinado de petróleo y su almacenamiento, así como las actividades de la banca extraterritorial. A pesar de los pobres suelos de la isla y la escasez de precipitaciones lo que limita sus posibilidades agrícolas, existen cultivos de aloe y actividades como la ganadería y la pesca contribuyen a la economía nacional. Además, el país también exporta arte y objetos de colección, maquinarias, equipos eléctricos y material de transporte. La escasa mano de obra de Aruba y la baja tasa de desempleo han dado lugar a un gran número de puestos de trabajo vacantes sin cubrir, a pesar de los fuertes aumentos en los salarios en los últimos años."
4,5,7733876#0,0.670403,,Banco Central de Aruba,"El Banco Central de Aruba () es el banco central en Aruba, responsable de la aplicación de la política monetaria del florín arubeño."
5,6,943022#4,0.669887,,Florín antillano neerlandés,"En 1986, Aruba ganó un estatus de autonomía y se separó de las Antillas Neerlandesas. En poco tiempo, Aruba comenzó a emitir su propia moneda, el florín arubeño, que sustituyó al florín de las Antillas Neerlandesas a un cambio de 1:1."
6,7,10403#56,0.658956,,Aruba,"Aruba es un país próspero. El desempleo es bajo (aunque el gobierno no publica estadísticas desde 2013) y la renta per cápita es una de las más altas del Caribe (aproximadamente 24.087 dólares). A finales de 2018, la tasa de participación en la fuerza laboral era del 56,6% para las mujeres."
7,8,3655473#11,0.655498,,Economía de Aruba,gastos: 577.9 millones de dólares (2005 estimado)
8,9,3655935#7,0.650602,,Idiomas de Aruba,"Aruba tiene 4 periódicos publicados en Papiamento: ""Diario"", ""Bon Dia"", ""Solo di Pueblo"" y ""Awe Mainta"", además de 2 en inglés: ""Aruba Today"" y ""The News"", 18 emisoras de radio (""2 AM y 16 FM"") y tres televisoras locales (Tele-Aruba, Aruba Broadcast Company y Star Television)."
9,10,9567892#43,0.644124,,Pandemia de COVID-19 en Aruba,"1 de mayo de 2020: el gobierno holandés aprobó un rescate 'suave' de 49,5 millones de florines (± 27,6 millones de dólares) para Aruba, que tuvo que reembolsarse en dos años sin intereses."


**jina-v5-small** — top 10 de «que moneda circula en aruba»

,posicion,docid,puntaje,relevante,titulo,extracto
0,1,328242#0,0.817917,sí,Florín arubeño,"El florín es la moneda oficial de Aruba. Se divide en 100 céntimos. Fue introducido en 1986, reemplazando a paridad el Florín antillano neerlandés."
1,2,7733876#2,0.788228,,Banco Central de Aruba,"El banco es una entidad legal en sí misma (""sui generis"") con una posición autónoma dentro del sector público de Aruba. Con la creación del banco, el florín arubeño fue puesto en circulación al mismo tipo de cambio que el florín antillano neerlandés, ligado al dólar estadounidense a Afl. 1.79 (= 1 NAf.) = US$1.00. Este tipo de cambio se ha mantenido sin cambios desde entonces."
2,3,328242#1,0.712161,,Florín arubeño,"Las monedas tienen denominaciones de 1, 2, 5 y 10 florines. Y los billetes tienen denominaciones de 10, 25, 50, 100, 250 y 500 florines."
3,4,10403#54,0.707369,,Aruba,"Alrededor del 70 % del PIB de Aruba proviene del turismo o de actividades relacionadas, y un 75 % de los visitantes procede de Estados Unidos. Antes de obtener su estatus autonómico, la mayor actividad era la refinación de petróleo; actualmente dicha actividad tiene una pequeña influencia en la economía. La agricultura y la manufactura tienen también un pequeño impacto económico. El florín arubiano generalmente tiene un cambio fijo con el dólar estadounidense de 1,75:1, pero en la mayoría de los comercios se aplica 1,80:1 como tasa de cambio. Sus principales socios comerciales son Venezuela, Estados Unidos, Países Bajos y Reino Unido."
4,5,943022#4,0.706751,,Florín antillano neerlandés,"En 1986, Aruba ganó un estatus de autonomía y se separó de las Antillas Neerlandesas. En poco tiempo, Aruba comenzó a emitir su propia moneda, el florín arubeño, que sustituyó al florín de las Antillas Neerlandesas a un cambio de 1:1."
5,6,5585611#0,0.698622,,Museo numismático de Aruba,"El Museo numismático de Aruba es un museo establecido en la ciudad de Oranjestad, la capital del país autónomo de los Países Bajos e isla caribeña de Aruba. Se trata de un espacio especializado que cuenta con una gran colección de monedas y billetes de todo el mundo, y que representa la pasión de un hombre local (Mario Odor) cuyo afición se transformó en una exhibición para el público. Fue creado oficialmente el 13 de noviembre de 1981, posee 35 piezas que vienen de 400 lugares."
6,7,7733876#0,0.675414,,Banco Central de Aruba,"El Banco Central de Aruba () es el banco central en Aruba, responsable de la aplicación de la política monetaria del florín arubeño."
7,8,328242#2,0.668061,,Florín arubeño,La moneda tiene una tasa de cambio con el dólar estadounidense de 1 US$ = 1.79 florines desde 1986.
8,9,68602#73,0.664497,,Bonaire,"En 2011, las Islas BES sustituyeron su moneda, el florín antillano neerlandés (ISO 4217: ANG, símbolo: ƒ), por el dólar estadounidense en lugar de sustituirlo por el euro que se utiliza en los Países Bajos europeos. La decisión se basó principalmente en las necesidades del turismo y el comercio. La mayoría de los países y territorios del Caribe utilizan el dólar como moneda o tienen una moneda vinculada al dólar como moneda de curso legal. El florín estuvo vinculado al dólar estadounidense durante décadas, con un tipo de cambio de 1,79 ƒ = 1,00 USD. La adopción del dólar puso fin al sistema de pago en dos monedas y a las comisiones por cambio de divisas. El florín siguió utilizándose en Curazao y Sint Maarten."
9,10,35612#0,0.663878,,Florín neerlandés,"El florín neerlandés () (conocido erróneamente como florín holandés) fue la moneda oficial de los Países Bajos desde el siglo XVII hasta 2002, cuando fue sustituido por el euro. Actualmente, el florín se utiliza todavía en las Antillas Neerlandesas y en Aruba, aunque estas monedas son distintas de las utilizadas en la metrópoli. En 2004, el florín de Surinam fue sustituido por el dólar surinamés."


**Documentos relevantes (verdad de referencia)**

328242#0 — Florín arubeño
El florín es la moneda oficial de Aruba. Se divide en 100 céntimos. Fue introducido en 1986, reemplazando a paridad el Florín antillano neerlandés.



## 7. Dónde aparece cada documento relevante

Para cada documento relevante de la consulta, la posición que le asigna cada modelo dentro de su top n (`—` si no aparece en él). La última columna cuenta en cuántos de los seis modelos aparece.

In [9]:
def resumen_relevantes(resultados, numero) -> pd.DataFrame:
    """Posición de cada documento relevante en el top n de cada modelo ('—' si no aparece)."""
    filas = []
    for docid in relevantes_por_consulta[numero]:
        fila = {"docid": docid}
        for alias, tabla in resultados.items():
            posiciones = tabla.index[tabla["docid"] == docid]
            fila[alias] = int(tabla.loc[posiciones[0], "posicion"]) if len(posiciones) else "—"
        fila["modelos"] = sum(1 for alias in resultados if fila[alias] != "—")
        filas.append(fila)
    return pd.DataFrame(filas)


display(resumen_relevantes(resultados, CONSULTA))

,docid,bm25,splade-v3,e5-large,bge-m3,qwen3-0.6b,jina-v5-small,modelos
0,328242#0,10,5,1,1,1,1,6


## Versión interactiva

Las secciones anteriores están atadas a la consulta de la §1 y dejan un ejemplo ejecutado en el cuaderno. Para inspeccionar **otras consultas** está `analizar_consulta(numero, n)`: recupera el top n de los seis modelos, carga los textos de esa consulta y muestra el resumen y las tablas. Tarda ≈30 s (leer los seis runs) y no hace falta volver a ejecutar nada de arriba.

`buscar_consultas` sirve para encontrar un número; en la última celda se cambia el número y se vuelve a ejecutar solo esa celda.

In [10]:
def analizar_consulta(numero, n: int | None = None) -> dict[str, pd.DataFrame]:
    """Recupera y muestra el top n de una consulta: la vía interactiva del cuaderno.

    Hace el recorrido completo para el número que reciba —top n por modelo, textos de esa
    consulta, resumen de los relevantes y tablas—, así que no depende de las variables de las
    secciones anteriores.

    Args:
        numero: número (id) de la consulta de prueba; ver `buscar_consultas`.
        n: cuántos documentos por modelo; por omisión, el `N` configurado en la §1.

    Returns:
        {alias: DataFrame}, los mismos resultados que muestra.
    """
    n = N if n is None else n
    numero = resolver_consulta(numero)
    resultados = top_n_por_modelo(numero, n)
    textos = cargar_textos(resultados, numero)

    display(Markdown(f"### Consulta {numero}: «{consulta_a_texto[numero]}»"))
    mostrar_relevantes(numero, textos)
    display(resumen_relevantes(resultados, numero))
    mostrar_resultados(resultados, numero, textos)
    return resultados

In [11]:
buscar_consultas("presidente")

7400945  a cuál batalla asistió el presidente mora
7423982  a quien propuso el rey como presidente
7430125  a qué presidente mexicano se le atribuye el apoyo para la apertura del primer periódico
7483566  como vuela el presidente de estados unidos
7494312  con que presidente inicia el neoliberalismo en mexico
7494314  con que presidente se dio el auge petrolero
7494315  con que presidente se inicio el neoliberalismo en mexico
7510540  cual es el presidente ruso
7512242  cual es el vicepresidente del ecuador
7516770  cual es la residencia oficial del presidente de la república dominicana
7520638  cual fue el presidente que duro menos tiempo en la presidencia
7520639  cual fue el presidente que duro menos tiempo en la presidencia argentina
7520753  cual fue el primer presidente de rd
7520878  cual fue el segundo presidente de estados unidos
7520880  cual fue el segundo presidente de usa
7521003  cual fue el ultimo presidente de la urss
7521004  cual fue el ultimo presidente militar
75217

In [12]:
# Cambia el número (y, si quieres, n) y vuelve a ejecutar solo esta celda.
NUMERO_CONSULTA = 7494312
N = 10

resultados = analizar_consulta(NUMERO_CONSULTA, N)

         bm25: 10 documentos en 4.7 s


    splade-v3: 10 documentos en 4.8 s


     e5-large: 10 documentos en 4.7 s


       bge-m3: 10 documentos en 4.7 s


   qwen3-0.6b: 10 documentos en 4.7 s


...

,posicion,docid,puntaje,relevante,titulo,extracto
0,1,10246424#6,0.669127,,Consecuencias laborales del neoliberalismo en México,"Se considera que De la Madrid fue el presidente que introdujo este modelo en México, esto es así porque con el comenzó la masiva privatización de empresas. De 1982 a 1988 pasaron de ser 1155 empresas gubernamentales a solo 412. Esto fue una medida que el presidente tomó para reducir los daños de la crisis en la que se encontraba el país cuando él tomó posesión de la presidencia, había mucha pobreza y desigualdad. Tal era la situación que la deuda externa se acrecentó, De la Madrid pagó 28 mmdd en deuda externa y, sin embargo, ésta aumentó, durante el sexenio, de 9 mil 400 millones de dólares en 1983 a 185 mil millones de dólares."
1,2,10246424#13,0.657214,,Consecuencias laborales del neoliberalismo en México,"El gobierno de este presidente se dio del año 2000 a 2006. Fue considerado en sus inicios un gobernante diferente ya que, prometía crecimientos anuales mayores a 7% y generación de empleos. Además, fue el primer presidente que no era del PRI, Fox era del PAN, y fue por eso que se le daba mayor expectativa."
2,3,101942#1,0.655858,,Miguel de la Madrid,"Durante su presidencia introdujo políticas neoliberales radicales para superar la crisis económica por la caída internacional de los precios del petróleo, iniciando una era de presidentes orientados al mercado en México, junto con medidas de austeridad que implicaban profundos recortes en el gasto público. A pesar de estas reformas, el crecimiento económico del país se mantuvo negativo con una alta inflación, mientras que los efectos sociales de las medidas de austeridad fueron particularmente duros para las clases media y baja, con los salarios reales cayendo a la mitad y con un fuerte aumento del desempleo y de la economía informal hacia el final de su gobierno."
3,4,1830#185,0.653503,,México,"En los últimos 25 años, en el marco de la denominada era del neoliberalismo, se han realizado en México reformas y ajustes estructurales significativos en la economía. Las primeras reformas económicas se realizaron entre 1989 y 1994, durante la administración de Carlos Salinas de Gortari, siendo la más importante y trascendente, por sus múltiples impactos en la estructura económica del país —unos positivos y otros negativos—, la controvertida negociación del acuerdo de libre comercio con Estados Unidos y Canadá (el Tratado de Libre Comercio de América del Norte, TLCAN), el cual entró en vigor el primer día de 1994, con lo que quedó oficialmente atrás el agotado modelo desarrollista de crecimiento de sustitución de importaciones, cuya época dorada se ubica en los años cincuenta y sesenta del siglo pasado, y prevaleció un modelo neoliberal orientado al “exterior”, promotor de las exportaciones. Las últimas reformas económicas son de factura reciente, y se realizaron entre 2013 y 2014, bajo la administración de Enrique Peña Nieto. Por su impacto potencial en la tasa de crecimiento, destacó la del sector energético, pues desde 2015 el sector privado, nacional y extranjero, participaba activamente en las tareas de exploración y explotación de petróleo crudo y gas, así como en la generación de energía eléctrica, actividades antes reservadas al Estado. Estas reformas estructurales, polémicas y controversiales, generaron —cada una en su momento— expectativas favorables respecto al crecimiento futuro de la economía mexicana, las cuales, sin embargo, no se materializaron del todo."
4,5,469947#80,0.639280,,Historia económica de México,Con la llegada de Andrés Manuel López Obrador en 2018 se terminó el período neoliberal con sus planes de volver a instaurar la intervención del Estado en la economía nacional con la creación de nuevas empresas estatales y programas sociales y el fortalecimiento de las empresas estatales existentes como lo son PEMEX y CFE.
5,6,4103#9,0.636432,,Economía de México,"El próximo presidente, Miguel de la Madrid, fue el primero en implementar una

**qwen3-0.6b** — top 10 de «con que presidente inicia el neoliberalismo en mexico»

,posicion,docid,puntaje,relevante,titulo,extracto
0,1,645953#439,0.754199,,Historia de México,"En este periodo comenzó la aplicación del modelo económico neoliberal, que retoma las ideas del liberalismo y considera negativo el intervencionismo estatal en la economía, al tiempo que defiende el libre mercado como la mejor opción del equilibrio y el desarrollo de los países; en el caso de México, la aplicación de este modelo —iniciada en este sexenio y profundizada en el de Carlos Salinas de Gortari—, se caracterizó por:"
1,2,10246424#13,0.738822,,Consecuencias laborales del neoliberalismo en México,"El gobierno de este presidente se dio del año 2000 a 2006. Fue considerado en sus inicios un gobernante diferente ya que, prometía crecimientos anuales mayores a 7% y generación de empleos. Además, fue el primer presidente que no era del PRI, Fox era del PAN, y fue por eso que se le daba mayor expectativa."
2,3,10246424#23,0.731791,,Consecuencias laborales del neoliberalismo en México,"El actual presidente significó un revuelo y críticas importantes de partidos como el PRI y el PAN ya que, sus discursos rechazaban las medidas Neoliberales que los anteriores presidentes promovieron en México. Además, mencionaba que durante su gobierno se daría una cuarta transformación."
3,4,10246424#3,0.727539,,Consecuencias laborales del neoliberalismo en México,"La principal razón por las que el neoliberalismo entró a México fue, como ya se mencionó antes, por la presión que generaron las crisis por todo el mundo y también dentro de México, quien pasaba por reducciones en el petróleo, un déficit fiscal, y una inflación de más de 100% (Quintana, 2016). Además, también se encontraba la devaluación del peso mexicano que poco a poco se volvía una amenaza para la estabilidad económica del país."
4,5,10246424#29,0.720661,,Consecuencias laborales del neoliberalismo en México,En el Cuadro 1 podemos ver que el crecimiento comenzó a reducirse a partir de que el neoliberalismo ingresó a México. El crecimiento se redujo a más del 50% del que había tenido en el periodo de 1940 a 1970. Fue a partir de Miguel de la Madrid que el PIB tuvo un crecimiento bajo en comparación con los años anteriores.
5,6,10246424#6,0.715605,,Consecuencias laborales del neoliberalismo en México,"Se considera que De la Madrid fue el presidente que introdujo este modelo en México, esto es así porque con el comenzó la masiva privatización de empresas. De 1982 a 1988 pasaron de ser 1155 empresas gubernamentales a solo 412. Esto fue una medida que el presidente tomó para reducir los daños de la crisis en la que se encontraba el país cuando él tomó posesión de la presidencia, había mucha pobreza y desigualdad. Tal era la situación que la deuda externa se acrecentó, De la Madrid pagó 28 mmdd en deuda externa y, sin embargo, ésta aumentó, durante el sexenio, de 9 mil 400 millones de dólares en 1983 a 185 mil millones de dólares."
6,7,10246424#5,0.698938,,Consecuencias laborales del neoliberalismo en México,"Para conocer sobre cómo se da este modelo neoliberal en México y las consecuencias que tuvo, es necesario conocer las modificaciones que se realizaron por sexenios a partir de 1982, año al que se le otorgó la entrada del neoliberalismo."
7,8,10697819#9,0.693958,,Neoliberalismo en el Perú,"El régimen neoliberal se instauró durante el gobierno de Alberto Fujimori, respaldado por la Confederación Nacional de Instituciones Empresariales Privadas (Confiep). Tras la desconfianza generada por la administración de Alan García, sin considerar el controvertido Plan Verde, el gobierno se centró en objetivos de estabilización (""Fujishock""), las reformas promercado y la inserción del Perú en el circuito financiero. En el marco de la reforma estatal, el economista Carlos Matus propuso la introducción de «tecnopolíticos» en puestos clave. Estos individuos poseían una perspectiva distinta y promovían valores democráticos y la satisfacción de las demandas ciudadanas."
8,9,10246424#2,0.691367,,Consecuenci

**jina-v5-small** — top 10 de «con que presidente inicia el neoliberalismo en mexico»

,posicion,docid,puntaje,relevante,titulo,extracto
0,1,10246424#6,0.714325,,Consecuencias laborales del neoliberalismo en México,"Se considera que De la Madrid fue el presidente que introdujo este modelo en México, esto es así porque con el comenzó la masiva privatización de empresas. De 1982 a 1988 pasaron de ser 1155 empresas gubernamentales a solo 412. Esto fue una medida que el presidente tomó para reducir los daños de la crisis en la que se encontraba el país cuando él tomó posesión de la presidencia, había mucha pobreza y desigualdad. Tal era la situación que la deuda externa se acrecentó, De la Madrid pagó 28 mmdd en deuda externa y, sin embargo, ésta aumentó, durante el sexenio, de 9 mil 400 millones de dólares en 1983 a 185 mil millones de dólares."
1,2,4103#9,0.708457,,Economía de México,"El próximo presidente, Miguel de la Madrid, fue el primero en implementar una serie de reformas de carácter neoliberal. Después de la crisis de 1982, pocas organizaciones internacionales estaban dispuestas a conceder préstamos a México, de modo que para alcanzar un balance de cuenta corriente ajustado, el gobierno recurrió a continuas devaluaciones, las cuales produjeron altos índices de inflación, que llegaron hasta el 159,7 % anual en 1987. Algunos efectos de las políticas de su administración fueron un incremento en el déficit público y el crédito interno."
2,3,67357#76,0.702847,sí,Neoliberalismo,"En 1988, tras una polémica elección presidencial, el economista Carlos Salinas de Gortari llegó a la Presidencia, imponiendo el modelo económico neoliberal en el país. La privatización de la banca se llevó a cabo mediante una reforma Constitucional a los artículos 28 y 123 que fueron aprobados el 12 de mayo de 1990 en la Cámara de Diputados y el 21 de mayo en el Senado. Guillermo Ortiz Martínez, Subsecretario de Hacienda con Salinas, fue uno de los responsables de este proceso."
3,4,1830#185,0.700830,,México,"En los últimos 25 años, en el marco de la denominada era del neoliberalismo, se han realizado en México reformas y ajustes estructurales significativos en la economía. Las primeras reformas económicas se realizaron entre 1989 y 1994, durante la administración de Carlos Salinas de Gortari, siendo la más importante y trascendente, por sus múltiples impactos en la estructura económica del país —unos positivos y otros negativos—, la controvertida negociación del acuerdo de libre comercio con Estados Unidos y Canadá (el Tratado de Libre Comercio de América del Norte, TLCAN), el cual entró en vigor el primer día de 1994, con lo que quedó oficialmente atrás el agotado modelo desarrollista de crecimiento de sustitución de importaciones, cuya época dorada se ubica en los años cincuenta y sesenta del siglo pasado, y prevaleció un modelo neoliberal orientado al “exterior”, promotor de las exportaciones. Las últimas reformas económicas son de factura reciente, y se realizaron entre 2013 y 2014, bajo la administración de Enrique Peña Nieto. Por su impacto potencial en la tasa de crecimiento, destacó la del sector energético, pues desde 2015 el sector privado, nacional y extranjero, participaba activamente en las tareas de exploración y explotación de petróleo crudo y gas, así como en la generación de energía eléctrica, actividades antes reservadas al Estado. Estas reformas estructurales, polémicas y controversiales, generaron —cada una en su momento— expectativas favorables respecto al crecimiento futuro de la economía mexicana, las cuales, sin embargo, no se materializaron del todo."
4,5,645953#439,0.698316,,Historia de México,"En este periodo comenzó la aplicación del modelo económico neoliberal, que retoma las ideas del liberalismo y considera negativo el intervencionismo estatal en la economía, al tiempo que defiende el libre mercado como la mejor opción del equilibrio y el desarrollo de los países; en el caso de México, la aplicación de este modelo —iniciada en este sexenio y profundizada en el de Carlos Salinas de Gortari—, se caracterizó 